# 🚀 vLLM Inference with LoRA Adapter for Iterative SFT
This notebook loads the Nemotron base model alongside your fine-tuned LoRA adapter. It evaluates the model on the full 9,500 problems and generates a comprehensive output CSV containing the model's reasoning, answers, and correctness.

## 📥 Step 1: Environment Setup & Load Model with LoRA Support

In [1]:
import subprocess
import sys

commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]

for cmd in commands:
    print(f"Running: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Prepend the unpacked environment directory to python search path
sys.path.insert(0, "/tmp")

Running: uv pip uninstall torch torchvision torchaudio


Using Python 3.12.13 environment at: /usr
Uninstalled 3 packages in 826ms
 - torch==2.10.0+cu128
 - torchaudio==2.10.0+cu128
 - torchvision==0.25.0+cu128


Running: tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell


In [2]:
import os
import zipfile
import kagglehub
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

# Set Triton and Transformers flags for offline execution
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# --- ADAPTER CONFIGURATION ---
# Set to True if loading from a .zip file (e.g., notebook output)
# Set to False if loading from a direct folder (e.g., Kaggle Dataset)
LOAD_FROM_ZIP = False

if LOAD_FROM_ZIP:
    ADAPTER_ZIP_PATH = "/kaggle/input/notebooks/huikang/tinker-submission-notebook/submission.zip" # <--- UPDATE THIS
    ADAPTER_PATH = "/tmp/lora_adapter"
    os.makedirs(ADAPTER_PATH, exist_ok=True)
    print(f"Extracting adapter from {ADAPTER_ZIP_PATH}...")
    with zipfile.ZipFile(ADAPTER_ZIP_PATH, "r") as zf:
        zf.extractall(ADAPTER_PATH)
else:
    # Path to the directory containing adapter_config.json and adapter_model.safetensors
    ADAPTER_PATH = "/kaggle/input/models/ckskaggle/nemo3-dapter-ck/transformers/default/1" # <--- UPDATE THIS

print(f"Using Base Model: {MODEL_PATH}")
print(f"Using extracted LoRA Adapter: {ADAPTER_PATH}")

# Initialize vLLM Engine WITH LoRA enabled
llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype="auto",
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=True,     # <--- ENABLED
    max_lora_rank=32      # <--- LoRA rank
)


2026-06-15 11:27:24.576117: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781522844.810871      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781522844.875193      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781522845.489171      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781522845.489186      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781522845.489187      65 computation_placer.cc:177] computation placer alr

Using Base Model: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Using extracted LoRA Adapter: /kaggle/input/models/ckskaggle/nemo3-dapter-ck/transformers/default/1
INFO 06-15 11:27:36 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 32, 'model': '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 06-15 11:28:01 [model.py:531] Resolved architecture: NemotronHForCausalLM
INFO 06-15 11:28:01 [model.py:1554] Using max model len 8192
INFO 06-15 11:28:01 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 06-15 11:28:01 [config.py:618] Updating mamba_ssm_cache_dtype to 'float32' for NemotronH model
INFO 06-15 11:28:01 [config.py:544] Setting attention block size to 2096 tokens to ensure that attention page size is >= mamba page size.
INFO 06-15 11:28:01 [config.py:575] Padding mamba page size by 0.58% to ensure that mamba page size and attention page size are exactly equal.
INFO 06-15 11:28:01 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 06-15 11:28:02 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


2026-06-15 11:28:08.229614: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781522888.239550     418 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781522888.242669     418 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781522888.250269     418 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781522888.250286     418 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781522888.250288     418 computation_placer.cc:177] computation placer alr

...

(EngineCore_DP0 pid=418) WARNING 06-15 11:36:23 [fused_moe.py:1093] Using default MoE config. Performance might be sub-optimal! Config file not found at /tmp/vllm/model_executor/layers/fused_moe/configs/E=128,N=1856,device_name=NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition.json
(EngineCore_DP0 pid=418) INFO 06-15 11:36:28 [backends.py:366] Compiling a graph for compile range (1, 16384) takes 10.66 s
(EngineCore_DP0 pid=418) INFO 06-15 11:36:28 [monitor.py:35] torch.compile takes 14.26 s in total
(EngineCore_DP0 pid=418) INFO 06-15 11:36:28 [decorators.py:580] saving AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/9f91a1aa0b2b51061b7ffc47be9f8301c731e59e5f3544f185fdc217a0d09037/rank_0_0/model
(EngineCore_DP0 pid=418) INFO 06-15 11:36:29 [decorators.py:588] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/9f91a1aa0b2b51061b7ffc47be9f8301c731e59e5f3544f185fdc217a0d09037/rank_0_0/model
(EngineCore_DP0 pid=418) INFO 06-15

(EngineCore_DP0 pid=418) 2026-06-15 11:36:34,276 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=418) 2026-06-15 11:36:34,489 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/38 [00:00<?, ?it/s]

(EngineCore_DP0 pid=418) WARNING 06-15 11:36:35 [utils.py:268] Using default LoRA kernel configs


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 38/38 [00:12<00:00,  3.15it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 22/22 [00:41<00:00,  1.88s/it]


(EngineCore_DP0 pid=418) INFO 06-15 11:37:29 [gpu_model_runner.py:5386] Graph capturing finished in 55 secs, took -3.62 GiB
(EngineCore_DP0 pid=418) INFO 06-15 11:37:29 [core.py:282] init engine (profile, create kv cache, warmup model) took 75.39 seconds
(EngineCore_DP0 pid=418) INFO 06-15 11:37:30 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 06-15 11:37:30 [llm.py:388] Supported tasks: ['generate']


## 💬 Step 2: Define Prompts and Chat Formatting

In [3]:
SYSTEM_PROMPT = """You are an expert pattern recognition solver competing in a reasoning challenge.

You will be given a puzzle from Alice's Wonderland where a secret transformation rule converts inputs to outputs.

Your approach MUST follow these steps inside <think> tags:
1. HYPOTHESIZE: Study each input-output example and state the rule clearly
2. VERIFY: Apply your rule to every example to confirm it holds
3. APPLY: Use the confirmed rule on the test input

Your final answer MUST be inside \boxed{} and nothing after it.

Format:
<think>
HYPOTHESIZE: ...
VERIFY: ...
APPLY: ...
</think>
\boxed{your_answer}"""

def build_prompt(puzzle_text: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": puzzle_text}
    ]


## 📁 Step 3: Load the inference_dataset.csv

In [4]:
import pandas as pd

# --- CONFIGURATION: Set to an integer (e.g. 10) for testing, or None for full dataset ---
MAX_SAMPLES = 10

# Load the dataset we prepared locally
data_path = "/kaggle/input/datasets/ckskaggle/nemotron-3-competition-datasets/inference_dataset.csv"
print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path, dtype=str)

if MAX_SAMPLES is not None:
    df = df.head(MAX_SAMPLES).copy()
    print(f"Sampling {MAX_SAMPLES} rows for testing...")

# Cast the boolean column properly
df['was_solved_in_training'] = df['was_solved_in_training'] == 'True'

print(f"Loaded {len(df)} problems.")
print(f"Solved in training: {df['was_solved_in_training'].sum()}")
print(f"Unsolved in training: {len(df) - df['was_solved_in_training'].sum()}")


Loading dataset from: /kaggle/input/datasets/ckskaggle/nemotron-3-competition-datasets/inference_dataset.csv
Sampling 10 rows for testing...
Loaded 10 problems.
Solved in training: 8
Unsolved in training: 2


## 🧪 Step 4: Test Single Inference (Raw Output)

In [5]:
# Grab the first prompt from the dataset
test_prompt = df.iloc[0]["prompt"]
print("--- TEST PROMPT ---")
print(test_prompt)
print("\n--- GENERATING ---")

messages = build_prompt(test_prompt)
tokenizer = llm.get_tokenizer()
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=7680
)

# Run a single generation
test_output = llm.generate(
    [input_text], 
    sampling_params,
    lora_request=LoRARequest("adapter", 1, ADAPTER_PATH)
)

print("\n--- RAW MODEL OUTPUT ---")
print(test_output[0].outputs[0].text)


--- TEST PROMPT ---
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100

--- GENERATING ---


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

WARNING 06-15 11:37:30 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


--- RAW MODEL OUTPUT ---
We need to deduce the transformation by matching the example outputs.

Output 1: 11011101
0 1
1 1
2 0
3 1
4 1
5 1
6 0
7 1

Output 2: 01101101
0 0
1 1
2 1
3 0
4 1
5 1
6 0
7 1

Output 3: 01010101
0 0
1 1
2 0
3 1
4 0
5 1
6 0
7 1

Output 4: 10000001
0 1
1 0
2 0
3 0
4 0
5 0
6 0
7 1

Output 5: 01000101
0 0
1 1
2 0
3 0
4 0
5 1
6 0
7 1

Output 6: 00001001
0 0
1 0
2 0
3 0
4 1
5 0
6 0
7 1

Output 7: 00000101
0 0
1 0
2 0
3 0
4 0
5 1
6 0
7 1

Output 8: 10110011
0 1
1 0
2 1
3 1
4 0
5 0
6 1
7 1

Output bit columns (with bitsum as hash)
0 10010001 3
1 11101000 4
2 01000001 2
3 10100001 3
4 11000100 3
5 11101010 5
6 00000001 1
7 11111111 a

Input 1: 01010001
0 0
1 1
2 0
3 1
4 0
5 0
6 0
7 1

Input 2: 00001001
0 0
1 0
2 0
3 0
4 1
5 0
6 0
7 1

Input 3: 00010101
0 0
1 0
2 0
3 1
4 0
5 1
6 0
7 1

Input 4: 11111111
0 1
1 1
2 1
3 1
4 1
5 1
6 1
7 1

Input 5: 10011101
0 1
1 0
2 0
3 1
4 1
5 1
6 0
7 1

Input 6: 00111011
0 0
1 0
2 1
3 1
4 1
5 0
6 1
7 1

Input 7: 10111101
0 1
1 0
2 1
3 1
4

## 🚀 Step 5: Run Batch Inference with LoRA

In [6]:
print(f"Preparing prompts for {len(df)} examples...")
tokenizer = llm.get_tokenizer()
prompts = []
for idx, row in df.iterrows():
    messages = build_prompt(row["prompt"])
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(input_text)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=7680
)

print(f"Running batch generation with LoRA...")
# Pass the LoRA request to the generate function
outputs = llm.generate(
    prompts, 
    sampling_params,
    lora_request=LoRARequest("adapter", 1, ADAPTER_PATH)
)
print("Batch inference complete!")


Preparing prompts for 10 examples...
Running batch generation with LoRA...


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batch inference complete!


## 📊 Step 6: Parse Outputs & Calculate Correctness

In [7]:
import re

def extract_boxed_content(text):
    if not text:
        return "NOT_FOUND"
    matches = re.findall(r"\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()
    return "NOT_FOUND"

def parse_model_output(output_text):
    thought = ""
    answer_raw = output_text
    
    if "</think>" in output_text:
        parts = output_text.split("</think>", 1)
        if "<think>" in parts[0]:
            sub_parts = parts[0].split("<think>", 1)
            thought = sub_parts[1].strip() if len(sub_parts) > 1 else sub_parts[0].strip()
        else:
            thought = parts[0].strip()
        answer_raw = parts[1].strip()
    
    final_ans = extract_boxed_content(answer_raw)
    return thought, final_ans

results = []
for i, out in enumerate(outputs):
    raw_text = out.outputs[0].text
    thought, model_answer = parse_model_output(raw_text)
    
    original_answer = str(df.iloc[i]["original_answer"]).strip()
    is_correct = (model_answer == original_answer)
    
    results.append({
        "id": df.iloc[i]["id"],
        "category": df.iloc[i]["category"],
        "prompt": df.iloc[i]["prompt"],
        "model_thought": thought,
        "model_answer": model_answer,
        "original_answer": original_answer,
        "was_solved_in_training": df.iloc[i]["was_solved_in_training"],
        "is_correct_now": is_correct
    })

df_results = pd.DataFrame(results)

# Calculate summary stats
total_acc = df_results["is_correct_now"].mean() * 100
solved_acc = df_results[df_results["was_solved_in_training"]]["is_correct_now"].mean() * 100
unsolved_acc = df_results[~df_results["was_solved_in_training"]]["is_correct_now"].mean() * 100

print(f"\n--- RESULTS ---")
print(f"Overall Accuracy: {total_acc:.2f}%")
print(f"Accuracy on PREVIOUSLY SOLVED problems: {solved_acc:.2f}%")
print(f"Accuracy on UNKNOWN/NEW problems: {unsolved_acc:.2f}%")

print(f"\n--- ACCURACY BY CATEGORY ---")
cat_acc = df_results.groupby('category')['is_correct_now'].mean() * 100
for cat, acc in cat_acc.items():
    print(f"{cat}: {acc:.2f}%")

# Save detailed results
out_file = "inference_results.csv"
df_results.to_csv(out_file, index=False)
print(f"\nDetailed results saved to {out_file}")



--- RESULTS ---
Overall Accuracy: 0.00%
Accuracy on PREVIOUSLY SOLVED problems: 0.00%
Accuracy on UNKNOWN/NEW problems: 0.00%

--- ACCURACY BY CATEGORY ---
bit_manipulation: 0.00%
cipher: 0.00%
cryptarithm_deduce: 0.00%
gravity: 0.00%
numeral: 0.00%
unit_conversion: 0.00%

Detailed results saved to inference_results.csv
